<a href="https://colab.research.google.com/github/niikun/ezkl/blob/main/notebooks/05_witness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. Witness（証拠）の生成

**前提ファイル**: `input.json`, `network.compiled`
（`01_onnx.ipynb`, `04_compile.ipynb` まで実行済み）

In [1]:
try:
    import google.colab, subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl", "onnx", "torch", "torchvision"])
except ImportError:
    pass

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import ezkl, json, os

compiled_model_path = '/content/drive/MyDrive/ezkl/network.compiled'
data_path           = '/content/drive/MyDrive/ezkl/input.json'
witness_path        = '/content/drive/MyDrive/ezkl/witness.json'

for f in [compiled_model_path, data_path]:
    assert os.path.exists(f), f"{f} がありません。"

## `input.json` の内容を確認

In [5]:
try:
    with open(data_path, 'r') as f:
        input_data_content = json.load(f)
    print("input.json の内容:")
    print(json.dumps(input_data_content, indent=2))
    if 'input_data' not in input_data_content:
        print("\nエラー: 'input_data' フィールドが input.json に見つかりませんでした。")
    else:
        print("\n'input_data' フィールドは input.json に存在します。")
except FileNotFoundError:
    print(f"エラー: {data_path} が見つかりません。")
except json.JSONDecodeError:
    print(f"エラー: {data_path} は有効なJSONファイルではありません。")


input.json の内容:
{
  "run_args": {
    "input_scale": 13,
    "param_scale": 13,
    "rebase_scale": null,
    "scale_rebase_multiplier": 1,
    "lookup_range": [
      0,
      0
    ],
    "logrows": 15,
    "num_inner_cols": 2,
    "variables": [
      [
        "batch_size",
        1
      ]
    ],
    "input_visibility": "Public",
    "output_visibility": "Public",
    "param_visibility": "Fixed",
    "rebase_frac_zero_constants": false,
    "check_mode": "UNSAFE",
    "decomp_base": 16384,
    "decomp_legs": 2,
    "bounded_log_lookup": false,
    "ignore_range_check_inputs_outputs": false,
    "epsilon": 2.220446049250313e-16,
    "disable_freivalds": false
  },
  "num_rows": 23396,
  "total_assignments": 46792,
  "total_const_size": 2111,
  "total_dynamic_col_size": 0,
  "max_dynamic_input_len": 0,
  "num_dynamic_lookups": 0,
  "num_shuffles": 0,
  "total_shuffle_col_size": 0,
  "einsum_params": {
    "equations": [
      [
        "abcde,abcde->abcd",
        {
          "a": 

## `input.json` を修正

In [6]:
import numpy as np

# Assuming a 1x1x28x28 input shape based on 'model_instance_shapes' in the output
# You need to replace this with your actual model input data
# For demonstration, we create a dummy input array
dummy_input = np.random.rand(1, 1, 28, 28).flatten().tolist()

# Add 'input_data' to the existing content
input_data_content['input_data'] = [dummy_input]

# Write the updated content back to input.json
with open(data_path, 'w') as f:
    json.dump(input_data_content, f, indent=2)

print(f"Updated {data_path} with 'input_data' field.")

# Verify the content again
with open(data_path, 'r') as f:
    updated_content = json.load(f)
print("\nUpdated input.json の内容 (一部):")
print(json.dumps(updated_content, indent=2))

if 'input_data' in updated_content:
    print("\n'input_data' フィールドは input.json に存在します。(修正後)")
else:
    print("\nエラー: 'input_data' フィールドが input.json に見つかりませんでした。(修正後)")

Updated /content/drive/MyDrive/ezkl/input.json with 'input_data' field.

Updated input.json の内容 (一部):
{
  "run_args": {
    "input_scale": 13,
    "param_scale": 13,
    "rebase_scale": null,
    "scale_rebase_multiplier": 1,
    "lookup_range": [
      0,
      0
    ],
    "logrows": 15,
    "num_inner_cols": 2,
    "variables": [
      [
        "batch_size",
        1
      ]
    ],
    "input_visibility": "Public",
    "output_visibility": "Public",
    "param_visibility": "Fixed",
    "rebase_frac_zero_constants": false,
    "check_mode": "UNSAFE",
    "decomp_base": 16384,
    "decomp_legs": 2,
    "bounded_log_lookup": false,
    "ignore_range_check_inputs_outputs": false,
    "epsilon": 2.220446049250313e-16,
    "disable_freivalds": false
  },
  "num_rows": 23396,
  "total_assignments": 46792,
  "total_const_size": 2111,
  "total_dynamic_col_size": 0,
  "max_dynamic_input_len": 0,
  "num_dynamic_lookups": 0,
  "num_shuffles": 0,
  "total_shuffle_col_size": 0,
  "einsum_params

## Witness とは

RareSkills では「**witness ベクトル `a`** = すべての制約を満たす変数の値」と学びました。

EZKL の witness は、実際の入力データを回路に通したときの：
- 入力値
- 各層の中間計算値
- 最終出力値

をすべて記録したものです。証明者はこの値を使って「正しく計算した」という証明（proof）を作ります。

In [7]:
res = ezkl.gen_witness(data_path, compiled_model_path, witness_path)
assert os.path.exists(witness_path)
print("Witness 生成完了")

Witness 生成完了


## witness.json の中身を確認する

In [8]:
with open(witness_path) as f:
    w = json.load(f)

print(f"キー: {list(w.keys())}\n")

if 'inputs' in w:
    for i, inp in enumerate(w['inputs']):
        print(f"inputs[{i}]  : {len(inp)} 個の値（入力画像のピクセル値）")

if 'outputs' in w:
    classes = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']
    for i, out in enumerate(w['outputs']):
        print(f"outputs[{i}] : {len(out)} 個の値（10クラスのスコア）")
        predicted = out.index(max(out))
        print(f"予測クラス  : [{predicted}] {classes[predicted]}")

キー: ['inputs', 'pretty_elements', 'outputs', 'processed_inputs', 'processed_params', 'processed_outputs', 'max_lookup_inputs', 'min_lookup_inputs', 'max_range_size', 'version']

inputs[0]  : 784 個の値（入力画像のピクセル値）
outputs[0] : 10 個の値（10クラスのスコア）
予測クラス  : [9] Ankle boot


In [ ]:
w

## Witness が必要な理由

```
Witness（計算証拠）
    + ZK 回路（制約の集まり）
    + 証明鍵 PK
         ↓
     proof.json（ZK 証明）
```

検証者は **proof だけ**で確認できます。Witness の中身（中間計算値）は検証に不要です。これが「ゼロ知識」の本質です。

---
次: `06_setup.ipynb` で SRS のダウンロードと鍵生成（Trusted Setup）を行います。